# LangChain Semantic Similarity Example Selectors Reference

# `sorted_values`
Returns dictionary values ordered by their sorted keys.

```python
sorted_values(
    values: dict[str, str], # Input-variable names mapped to their values
) -> list[str] # Values ordered by sorted key
```

# `SemanticSimilarityExampleSelector: _VectorStoreExampleSelector`

Selects examples whose vector-store representations are most semantically similar to the current input.

## Fields

```python
vectorstore: VectorStore # Vector store containing example text and metadata
k: int = 4 # Number of examples to select
example_keys: list[str] | None = None # Keys retained in returned examples
input_keys: list[str] | None = None # Keys used to build example and query text
vectorstore_kwargs: dict[str, Any] | None = None # Extra arguments passed to similarity_search()
```

## Constructor

```python
SemanticSimilarityExampleSelector(
    *,
    vectorstore: VectorStore, # Vector store containing the examples
    k: int = 4, # Number of examples to select
    example_keys: list[str] | None = None, # Keys retained in returned examples
    input_keys: list[str] | None = None, # Keys used to build search text
    vectorstore_kwargs: dict[str, Any] | None = None, # Extra similarity-search arguments
) -> None
```

Extra constructor fields are forbidden. Arbitrary vector-store types are allowed.

## Methods

### `add_example`

Adds an example to the vector store and returns its generated ID.

```python
add_example(
    self,
    example: dict[str, str], # Example values keyed by input-variable name
) -> str # ID returned for the added example
```

The example text is formed by joining values in sorted-key order. When `input_keys` is provided, only those keys contribute to the stored text. The complete example is stored as metadata.

### `aadd_example`

Asynchronously adds an example through `VectorStore.aadd_texts()`.

```python
async aadd_example(
    self,
    example: dict[str, str], # Example values keyed by input-variable name
) -> str # ID returned for the added example
```

### `select_examples`

Returns the most similar examples for the supplied input variables.

```python
select_examples(
    self,
    input_variables: dict[str, str], # Current input values used to build the search query
) -> list[dict[str, Any]] # Selected examples reconstructed from document metadata
```

The query text uses values ordered by sorted key, restricted to `input_keys` when configured. The method calls `vectorstore.similarity_search()` with `k` and the contents of `vectorstore_kwargs`.

Returned examples are copied from document metadata. When `example_keys` is configured, only those keys are retained.

### `aselect_examples`

Asynchronously selects examples through `VectorStore.asimilarity_search()`.

```python
async aselect_examples(
    self,
    input_variables: dict[str, str], # Current input values used to build the search query
) -> list[dict[str, Any]] # Selected examples reconstructed from document metadata
```

### `from_examples`

Creates a selector and populates a vector store from an example list.

```python
@classmethod
from_examples(
    cls,
    examples: list[dict[str, str]], # Examples used to populate the vector store
    embeddings: Embeddings, # Embedding implementation used by the vector store
    vectorstore_cls: type[VectorStore], # Vector-store class constructed with from_texts()
    k: int = 4, # Number of examples to select
    input_keys: list[str] | None = None, # Keys used to build stored and query text
    *,
    example_keys: list[str] | None = None, # Keys retained in returned examples
    vectorstore_kwargs: dict[str, Any] | None = None, # Extra similarity-search arguments
    **vectorstore_cls_kwargs: Any, # Arguments forwarded to vectorstore_cls.from_texts()
) -> SemanticSimilarityExampleSelector # Populated selector
```

The vector store receives the generated example strings and the original examples as metadata.

### `afrom_examples`

Asynchronously creates a selector through `vectorstore_cls.afrom_texts()`.

```python
@classmethod
async afrom_examples(
    cls,
    examples: list[dict[str, str]], # Examples used to populate the vector store
    embeddings: Embeddings, # Embedding implementation used by the vector store
    vectorstore_cls: type[VectorStore], # Vector-store class constructed with afrom_texts()
    k: int = 4, # Number of examples to select
    input_keys: list[str] | None = None, # Keys used to build stored and query text
    *,
    example_keys: list[str] | None = None, # Keys retained in returned examples
    vectorstore_kwargs: dict[str, Any] | None = None, # Extra similarity-search arguments
    **vectorstore_cls_kwargs: Any, # Arguments forwarded to vectorstore_cls.afrom_texts()
) -> SemanticSimilarityExampleSelector # Populated selector
```

In [ ]:
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.example_selectors import SemanticSimilarityExampleSelector # Import the selector
from langchain_core.vectorstores import InMemoryVectorStore # Import the in-memory vector store


class KeywordEmbeddings(Embeddings): # Create simple embeddings for demonstration
    keywords = ["python", "list", "tuple", "dictionary", "sql", "group", "pandas", "axis"] # Define searchable words

    def embed_query(self, text: str) -> list[float]: # Convert one text into a vector
        text = text.lower() # Make matching case-insensitive
        return [float(text.count(word)) for word in self.keywords] # Count each keyword

    def embed_documents(self, texts: list[str]) -> list[list[float]]: # Convert many texts
        return [self.embed_query(text) for text in texts] # Reuse embed_query


examples = [ # Create examples
    {
        "input": "What is a Python list?",
        "output": "A list is a mutable collection.",
    }, # Add a Python list example
    {
        "input": "What is a Python tuple?",
        "output": "A tuple is an immutable collection.",
    }, # Add a Python tuple example
    {
        "input": "What is SQL GROUP BY?",
        "output": "GROUP BY creates groups of rows.",
    }, # Add an SQL example
    {
        "input": "What does Pandas axis mean?",
        "output": "Axis controls the operation direction.",
    }, # Add a Pandas example
] # Finish the example list

selector = SemanticSimilarityExampleSelector.from_examples( # Create the selector
    examples=examples, # Provide the examples
    embeddings=KeywordEmbeddings(), # Provide the embedding model
    vectorstore_cls=InMemoryVectorStore, # Store vectors in memory
    k=2, # Return two similar examples
    input_keys=["input"], # Use only input for searching
    example_keys=["input", "output"], # Return only these fields
) # Finish creating the selector

selected = selector.select_examples( # Select examples synchronously
    {"input": "Explain a Python list"} # Provide the search input
) # Finish selecting examples

print("Selected examples:") # Display a heading

for example in selected: # Visit each selected example
    print(example) # Display the example

new_id = selector.add_example( # Add a new example
    {
        "input": "What is a Python dictionary?",
        "output": "A dictionary stores key-value pairs.",
    }
) # Finish adding the example

print("\nAdded example ID:", new_id) # Display the generated ID

async_selected = await selector.aselect_examples( # Search asynchronously in Jupyter
    {"input": "Explain a Python dictionary"} # Provide another search input
) # Finish asynchronous selection

print("\nAsynchronously selected examples:") # Display a heading

for example in async_selected: # Visit each selected example
    print(example) # Display the example

# `MaxMarginalRelevanceExampleSelector: _VectorStoreExampleSelector`

Selects examples through maximal marginal relevance, balancing query relevance and diversity among the selected examples.

## Fields

```python
vectorstore: VectorStore # Vector store containing example text and metadata
k: int = 4 # Number of examples to select
example_keys: list[str] | None = None # Keys retained in returned examples
input_keys: list[str] | None = None # Keys used to build example and query text
vectorstore_kwargs: dict[str, Any] | None = None # Stored vector-store keyword arguments
fetch_k: int = 20 # Number of candidate documents fetched for MMR reranking
```

## Constructor

```python
MaxMarginalRelevanceExampleSelector(
    *,
    vectorstore: VectorStore, # Vector store containing the examples
    k: int = 4, # Number of examples to select
    example_keys: list[str] | None = None, # Keys retained in returned examples
    input_keys: list[str] | None = None, # Keys used to build search text
    vectorstore_kwargs: dict[str, Any] | None = None, # Stored vector-store keyword arguments
    fetch_k: int = 20, # Number of candidates fetched for reranking
) -> None
```

Extra constructor fields are forbidden. Arbitrary vector-store types are allowed.

## Methods

### `add_example`

Adds an example to the vector store and returns its generated ID.

```python
add_example(
    self,
    example: dict[str, str], # Example values keyed by input-variable name
) -> str # ID returned for the added example
```

### `aadd_example`

Asynchronously adds an example through `VectorStore.aadd_texts()`.

```python
async aadd_example(
    self,
    example: dict[str, str], # Example values keyed by input-variable name
) -> str # ID returned for the added example
```

### `select_examples`

Selects examples through maximal marginal relevance.

```python
select_examples(
    self,
    input_variables: dict[str, str], # Current input values used to build the search query
) -> list[dict[str, Any]] # Selected examples reconstructed from document metadata
```

The method calls `vectorstore.max_marginal_relevance_search()` with `k` and `fetch_k`. The stored `vectorstore_kwargs` value is not forwarded by this implementation.

Returned examples are copied from document metadata and filtered to `example_keys` when configured.

### `aselect_examples`

Asynchronously selects examples through `VectorStore.amax_marginal_relevance_search()`.

```python
async aselect_examples(
    self,
    input_variables: dict[str, str], # Current input values used to build the search query
) -> list[dict[str, Any]] # Selected examples reconstructed from document metadata
```

### `from_examples`

Creates an MMR selector and populates a vector store from examples.

```python
@classmethod
from_examples(
    cls,
    examples: list[dict[str, str]], # Examples used to populate the vector store
    embeddings: Embeddings, # Embedding implementation used by the vector store
    vectorstore_cls: type[VectorStore], # Vector-store class constructed with from_texts()
    k: int = 4, # Number of examples to select
    input_keys: list[str] | None = None, # Keys used to build stored and query text
    fetch_k: int = 20, # Number of candidates fetched for MMR reranking
    example_keys: list[str] | None = None, # Keys retained in returned examples
    vectorstore_kwargs: dict[str, Any] | None = None, # Stored vector-store keyword arguments
    **vectorstore_cls_kwargs: Any, # Arguments forwarded to vectorstore_cls.from_texts()
) -> MaxMarginalRelevanceExampleSelector # Populated selector
```

### `afrom_examples`

Asynchronously creates an MMR selector through `vectorstore_cls.afrom_texts()`.

```python
@classmethod
async afrom_examples(
    cls,
    examples: list[dict[str, str]], # Examples used to populate the vector store
    embeddings: Embeddings, # Embedding implementation used by the vector store
    vectorstore_cls: type[VectorStore], # Vector-store class constructed with afrom_texts()
    *,
    k: int = 4, # Number of examples to select
    input_keys: list[str] | None = None, # Keys used to build stored and query text
    fetch_k: int = 20, # Number of candidates fetched for MMR reranking
    example_keys: list[str] | None = None, # Keys retained in returned examples
    vectorstore_kwargs: dict[str, Any] | None = None, # Stored vector-store keyword arguments
    **vectorstore_cls_kwargs: Any, # Arguments forwarded to vectorstore_cls.afrom_texts()
) -> MaxMarginalRelevanceExampleSelector # Populated selector
```

In [ ]:
from langchain_core.embeddings import Embeddings # Import the embedding interface
from langchain_core.example_selectors import MaxMarginalRelevanceExampleSelector # Import the MMR selector
from langchain_core.vectorstores import InMemoryVectorStore # Import the in-memory vector store


class KeywordEmbeddings(Embeddings): # Create a simple embedding model for demonstration
    keywords = [ # Define words represented in each vector
        "python",
        "list",
        "tuple",
        "dictionary",
        "collection",
        "sql",
        "group",
        "pandas",
        "axis",
    ]

    def embed_query(self, text: str) -> list[float]: # Convert one text into a vector
        text = text.lower() # Make keyword matching case-insensitive
        return [float(text.count(word)) for word in self.keywords] # Count every keyword

    def embed_documents(self, texts: list[str]) -> list[list[float]]: # Convert many texts
        return [self.embed_query(text) for text in texts] # Reuse the query embedding method


examples = [ # Create examples for selection
    {
        "input": "What is a Python list?",
        "output": "A list is a mutable collection.",
        "topic": "Python",
    },
    {
        "input": "How do I add an item to a Python list?",
        "output": "Use the append method.",
        "topic": "Python",
    },
    {
        "input": "What is a Python tuple?",
        "output": "A tuple is an immutable collection.",
        "topic": "Python",
    },
    {
        "input": "What is a Python dictionary?",
        "output": "A dictionary stores key-value pairs.",
        "topic": "Python",
    },
    {
        "input": "What is SQL GROUP BY?",
        "output": "GROUP BY creates groups of rows.",
        "topic": "SQL",
    },
] # Finish the example list

selector = MaxMarginalRelevanceExampleSelector.from_examples( # Create the selector
    examples=examples, # Provide examples to store
    embeddings=KeywordEmbeddings(), # Provide the embedding model
    vectorstore_cls=InMemoryVectorStore, # Store vectors in memory
    k=2, # Return two examples
    fetch_k=5, # Fetch five candidates before MMR reranking
    input_keys=["input"], # Use only input text for similarity
    example_keys=["input", "output"], # Return only input and output fields
) # Finish creating the selector

selected_examples = selector.select_examples( # Select relevant and diverse examples
    {"input": "Explain Python collections"} # Provide the current input
) # Finish selecting examples

print("Selected examples:") # Display a heading

for example in selected_examples: # Visit each selected example
    print(example) # Display the example

new_id = selector.add_example( # Add another example
    {
        "input": "What is a Pandas axis?",
        "output": "Axis specifies the direction of an operation.",
        "topic": "Pandas",
    }
) # Finish adding the example

print("\nAdded example ID:", new_id) # Display the generated ID

async_examples = await selector.aselect_examples( # Select examples asynchronously in Jupyter
    {"input": "Explain Pandas axis"} # Provide another input
) # Finish asynchronous selection

print("\nAsynchronously selected examples:") # Display another heading

for example in async_examples: # Visit each asynchronous result
    print(example) # Display the selected example